# Goal

Тестируем  `18d_world_model_02`, а именно:
1) отчуждаемый `VisionHead`
2) факторизованный `spatial_encoding`
3) ещё раз `g_encoding.is_common`. В прошлом исследовании, как оказалось была ошибка (упоминался is_shared, а не из is_common). 

# GRID_SEARCH_SPACE

In [ ]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [25]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.system.comment = None
    HP.system.random_seed = random.randint(0, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    HP.system.use_amp = True
    
    HP.dataset.train = [
        'train_dataset:100',
        'train_dataset:101',
        'train_dataset:102',
        'train_dataset:103',
        'train_dataset:104',
        'train_dataset:105',
        'train_dataset:106',
        'train_dataset:107',
        'train_dataset:108',
        'train_dataset:109',
    ]
    HP.dataset.test = 'test_dataset:2'

    HP.model.parent = None
    HP.model.is_reconstruction = True 
    HP.model.is_prediction = True
    HP.model.sequence_length = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.vision_head = dict(grid=(6,6), features_counts=(16, 32, 64, 128))
    HP.model.g_encoding = dict(is_common=random.randint(0, 1) == 1)
    HP.model.d_model = 256
    HP.model.layers_count = 3
    HP.model.heads_count = 4
    HP.model.attention_backend = 'EFFICIENT_ATTENTION' # MATH, EFFICIENT_ATTENTION, FLASH_ATTENTION
    HP.model.render_heads_count = 4
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0005)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    
    return HP
# @launchit.stop

# Results
<TBD>

Видно, что `g_encoding.is_common=False` показывает лучшие результаты. Удивительно, т.к. я думал, что регуляризация (`is_common=True`) будет лучшее. 

<img src="./img/launches.png">

**Выводы**
1) отчуждаемый `VisionHead` и факторизованный `spatial_encoding` работают
2) проверить на дообучении, как будут вести себя `g_encoding.is_common=False` и `g_encoding.is_common=True`